In [ ]:
import pandas as pd
import numpy as np
import os
import h5py
from scipy.io import mmread
from scipy.sparse import csr_matrix

# Load Preprocessed Data

## Define Target Directory

In [ ]:
dataset = "Xe_hsa_breast_c"  # Set the actual dataset identifier.
outdir = "PATH/TO/RESULT/filtered_feature_bc_matrix"
os.makedirs(outdir, exist_ok=True)
os.makedirs(f"{outdir}/spatial", exist_ok=True)

## Load Gene Expression Matrix

In [ ]:
exp_data=pd.read_csv(
    "PATH/TO/cnts.tsv",
    sep="\t",
    index_col=0
)
exp_data

## Load Barcode information

If the location file has been adjusted to a standard tissue position file format, with `in_tissue`, `array_row`, `array_col`, `pxl_col_in_fullres`, and `pxl_row_in_fullres` included.

In [ ]:
locs_raw = pd.read_csv(
    "/PATH/TO/locs-raw.tsv",
    sep='\t',

    header=0, # if the first row is the header 
    index_col=0
)
locs = locs_raw.copy()


locs

If the location file only has pixel scale coordinate information and lacks other columns

In [ ]:
locs_raw = pd.read_csv(
    "PATH/TO/locs-raw.tsv",
    sep='\t',

    header=0, # if the first row is the header 
    index_col=0
)

# generate `in_tissue` colunm
locs_raw['in_tissue'] = 1

# generate `array_row` and `array_col` colunm
locs_raw['array_row'] = locs_raw['y']
locs_raw['array_col'] = locs_raw['x']
x = list(locs_raw['x'].values)
y = list(locs_raw['y'].values)
x_keys_sort = sorted(set(x))
y_keys_sort = sorted(set(y))
x_rank_dict = {value: rank for rank, value in enumerate(x_keys_sort)}
y_rank_dict = {value: rank for rank, value in enumerate(y_keys_sort)}
locs_raw = locs_raw.replace({"array_row": y_rank_dict, "array_col": x_rank_dict})
locs_raw["array_row"] = locs_raw["array_row"].astype(int)
locs_raw["array_col"] = locs_raw["array_col"].astype(int)
locs_raw.rename(columns={'x':'pxl_col_in_fullres','y':'pxl_row_in_fullres'},inplace = True)
locs = locs_raw[['in_tissue','array_row','array_col','pxl_row_in_fullres','pxl_col_in_fullres']]
locs.index.name = None
locs

Writes the spot location information to the tissue_positions_list file. Note that the real-world file does not contain headers. The version saved here with the header is for viewing only.

In [ ]:
locs.to_csv(
    f"{outdir}/spatial/tissue_positions_list_with_header.csv", 
    sep=',', 
    index=True,
    header=True
)

locs.to_csv(
    f"{outdir}/spatial/tissue_positions_list.csv", 
    sep=',', 
    index=True,
    header=False
)

## Load Feature Information

Set `file_path` to the original Xenium feature H5, or to `None` when unavailable. In the latter case, gene names are used as feature IDs. Counts and their column order are retained.


In [ ]:
file_path = None  # Optional original Xenium cell_feature_matrix.h5.
if file_path is None:
    features_raw_data = pd.DataFrame({"id": exp_data.columns, "name": exp_data.columns,
                                      "feature_type": "Gene Expression"})
else:
    with h5py.File(file_path, 'r') as f:
        features_group = f['/matrix/features']
        gene_ids = features_group['id'][:]
        gene_names = features_group['name'][:]
        feature_type = features_group['feature_type'][:]
        
        gene_ids = [x.decode('utf-8') if isinstance(x, bytes) else x for x in gene_ids]
        gene_names = [x.decode('utf-8') if isinstance(x, bytes) else x for x in gene_names]
        feature_type = [x.decode('utf-8') if isinstance(x, bytes) else x for x in feature_type]
    
    features_raw_data = pd.DataFrame({'id': gene_ids, 'name': gene_names, 'feature_type': feature_type})
    features_raw_data


In [ ]:
features_use = pd.DataFrame(exp_data.columns)
features_use.columns = ['name']
features_use

In [ ]:
features_raw_data = features_raw_data.drop_duplicates(subset=['name'], keep='first')
features = features_raw_data.set_index('name').reindex(exp_data.columns).rename_axis('name').reset_index()
if features['id'].isna().any():
    raise ValueError("Missing feature metadata for expression-matrix genes")
features['genome'] = "GRCh38"
features = features[['id', 'name', 'feature_type', 'genome']]
features

## Generate matrix.mtx.gz barcodes.tsv.gz and features.tsv.gz

In [ ]:
import os
import shutil 
import gzip
import scipy
import time

##save matrix.mtx.gz
reAnno_count_sparse_mtx = scipy.sparse.csr_matrix(exp_data.T.values)
with open(os.path.join(outdir, 'matrix.mtx'), "wb") as matrix_file:
    scipy.io.mmwrite(matrix_file, reAnno_count_sparse_mtx)
with open(os.path.join(outdir,'matrix.mtx'),'rb') as mtx_in:
        with gzip.open(os.path.join(outdir,'matrix.mtx') + '.gz','wb') as mtx_gz:
            shutil.copyfileobj(mtx_in, mtx_gz)
os.remove(os.path.join(outdir,'matrix.mtx'))
##save barcodes.tsv.gz
barcodesFile = pd.DataFrame(exp_data.index)
barcodesFile.to_csv(os.path.join(outdir,"barcodes.tsv.gz"),sep='\t',header =False,index=False)
##save features.tsv.gz
features.to_csv(os.path.join(outdir,"features.tsv.gz"),sep='\t',header =False,index=False)

# Generate H5 File

## Load Gene Expression Matrix

In [ ]:
expression_data = mmread(f"{outdir}/matrix.mtx.gz").tocsc()
expression_data

## Load Feature

In [ ]:
features = pd.read_csv(f"{outdir}/features.tsv.gz", header=None, sep="\t")
print(features)
# rename columns
features.columns = ["id", "name", "feature_type","genome"]
print(features)

## Load Barcode

In [ ]:
barcodes = pd.read_csv(f"{outdir}/barcodes.tsv.gz", header=None)
barcodes.columns = ["barcode"]
barcodes

## Write Information to H5 File

In [ ]:
# Load information from files
barcode_list = barcodes["barcode"].values
gene_id_list = features["id"].values
gene_name_list = features["name"].values
gene_type_list = features["feature_type"].values
gene_genome_list = features["genome"]

# generate H5 file
with h5py.File(f"{outdir}/filtered_feature_bc_matrix.h5", 'w') as f:
# Add library_ids to attrs
    f.attrs['library_ids'] = np.array([dataset], dtype='S')
    
# Add Matrix group
    group = f.create_group('matrix')
## Add data
    group.create_dataset('data', data=expression_data.data)
    group.create_dataset('indices', data=expression_data.indices)
    group.create_dataset('indptr', data=expression_data.indptr)
    group.create_dataset('shape', data=expression_data.shape)
    group.create_dataset('barcodes', data=np.array(barcode_list, dtype='S'))
## Add feature group
    feature_group = group.create_group('features')
### Add data
    feature_group.create_dataset('id', data=np.array(gene_id_list, dtype='S'))
    feature_group.create_dataset('name', data=np.array(gene_name_list, dtype='S'))
    feature_group.create_dataset('feature_type', data=np.array(gene_type_list, dtype='S'))
    feature_group.create_dataset('genome', data=np.array(gene_genome_list, dtype='S'))
    feature_group.create_dataset('_all_tag_keys', data=np.array([b'genome'], dtype='S'))
### Add target group
    target_group = feature_group.create_group('target_sets')
    target_group.create_dataset('AddOn Panel', data=np.arange(len(features), dtype=np.int64))

# Inspect the exported H5


Inspect the stored matrix fields and dimensions.

In [ ]:
import h5py
with h5py.File(f"{outdir}/filtered_feature_bc_matrix.h5","r") as f:
    print(f['matrix'].keys())

In [ ]:
with h5py.File(f"{outdir}/filtered_feature_bc_matrix.h5", "r") as handle:
    print("Gene x spot shape:", tuple(handle["matrix/shape"][:]))